In [1]:
import pandas as pd
import numpy as np

import time

from PIL import Image
import ot
import os
import glob
import matplotlib.pyplot as plt


# Load images

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════════════════════════════════════
IMAGE_DIR        = 'Data/images' # 'saxs_routeB-v3/images'
OUTPUT_DIR       = 'Data'
INTENSITY_THRESH = -1
N_PROJECTIONS    = 50
TARGET_SIZE      = (128, 128)
N_IMAGES         = 500    
# ══════════════════════════════════════════════════════════════════════════════

os.makedirs(OUTPUT_DIR, exist_ok=True)


def load_image_as_2d_distribution(image_path, target_size=(128, 128),
                                   intensity_threshold=0,
                                   normalize_coords=True):
    img  = Image.open(image_path).convert('L')
    # img  = img.resize(target_size, Image.LANCZOS)
    arr  = np.array(img, dtype=np.float64)
    H, W = arr.shape

    rows, cols  = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
    coords      = np.stack([rows.ravel(), cols.ravel()], axis=1).astype(np.float64)
    intensities = arr.ravel()

    mask    = intensities > intensity_threshold
    coords  = coords[mask]
    weights = intensities[mask]

    if weights.sum() == 0:
        raise ValueError(f"No pixels above threshold in {image_path}")

    weights /= weights.sum()

    return coords, weights


# ── Load images ───────────────────────────────────────────────────────────────
all_paths   = sorted(glob.glob(os.path.join(IMAGE_DIR, '*.png')))
if not all_paths:
    raise FileNotFoundError(f"No PNG files found in {IMAGE_DIR}")

image_paths = all_paths[:N_IMAGES]

print(f"Found {len(all_paths)} images total — using first {len(image_paths)}\n")

# ── Build distributions ───────────────────────────────────────────────────────
print("Loading images as 2D distributions...")
distributions = []

for i, path in enumerate(image_paths):
    coords, weights = load_image_as_2d_distribution(
        path,
        target_size=TARGET_SIZE,
        intensity_threshold=INTENSITY_THRESH
    )
    distributions.append((coords, weights))
    print(f"  [{i:>3}] {os.path.basename(path):45s} → {len(weights):>8,} support points")

n     = len(distributions)
total = n * (n - 1) // 2

print(f"\n Loaded {n} images.")
print(f"   Total pairs        : {total:,}")
print(f"   Cost matrix/pair   : {TARGET_SIZE[0]*TARGET_SIZE[1]:,} × "
      f"{TARGET_SIZE[0]*TARGET_SIZE[1]:,} ≈ "
      f"{(TARGET_SIZE[0]*TARGET_SIZE[1])**2 * 8 / 1e6:.0f} MB")
print("Done")

Found 500 images total — using first 500

Loading images as 2D distributions...
  [  0] saxsB_0000.png                                →    4,096 support points
  [  1] saxsB_0001.png                                →    4,096 support points
  [  2] saxsB_0002.png                                →    4,096 support points
  [  3] saxsB_0003.png                                →    4,096 support points
  [  4] saxsB_0004.png                                →    4,096 support points
  [  5] saxsB_0005.png                                →    4,096 support points
  [  6] saxsB_0006.png                                →    4,096 support points
  [  7] saxsB_0007.png                                →    4,096 support points
  [  8] saxsB_0008.png                                →    4,096 support points
  [  9] saxsB_0009.png                                →    4,096 support points
  [ 10] saxsB_0010.png                                →    4,096 support points
  [ 11] saxsB_0011.png                  

# Compute distances

In [11]:
wd_filename   = 'wasserstein_distances_full.npy'
swd_filename  = 'sliced_wasserstein_distances_full.npy'
mask_filename = 'completed_mask.npy'

if os.path.exists(wd_filename) and os.path.exists(swd_filename) and os.path.exists(mask_filename):
    WD_matrix  = np.load(wd_filename)
    SWD_matrix = np.load(swd_filename)
    done_mask  = np.load(mask_filename)
    print(f"Loaded existing matrices, resuming... ({done_mask.sum()//2} pairs already done)")
else:
    WD_matrix  = np.full((n, n), np.nan)
    SWD_matrix = np.full((n, n), np.nan)
    done_mask  = np.zeros((n, n), dtype=bool)
    print("Starting fresh...")

completed = 0
for i in range(n):
    for j in range(i + 1, n):
        if done_mask[i, j]:
            continue

        wd, swd = compute_pair(images, i, j, n_projections=N_PROJECTIONS)

        WD_matrix[i, j]  = WD_matrix[j, i]  = wd
        SWD_matrix[i, j] = SWD_matrix[j, i] = swd
        done_mask[i, j]  = done_mask[j, i]  = True
        completed += 1

        if completed % 200 == 0:
            print(f"[{completed}/{total}] WD={wd:.6f}  SWD={swd:.6f}")
            np.save(wd_filename,  WD_matrix)
            np.save(swd_filename, SWD_matrix)
            np.save(mask_filename, done_mask)

np.save(wd_filename,  WD_matrix)
np.save(swd_filename, SWD_matrix)
np.save(mask_filename, done_mask)
print(f"Done. {completed} pairs computed this run.")

Loaded existing matrices, resuming... (60600 pairs already done)
[200/124750] WD=0.091197  SWD=0.037775
[400/124750] WD=0.053802  SWD=0.021544
[600/124750] WD=0.136711  SWD=0.062881
[800/124750] WD=0.203423  SWD=0.094255
[1000/124750] WD=0.168516  SWD=0.077903
[1200/124750] WD=0.108616  SWD=0.046523
[1400/124750] WD=0.074217  SWD=0.030848
[1600/124750] WD=0.132724  SWD=0.063489
[1800/124750] WD=0.129925  SWD=0.057535
[2000/124750] WD=0.053229  SWD=0.020715
[2200/124750] WD=0.046605  SWD=0.018979
[2400/124750] WD=0.066319  SWD=0.027296
[2600/124750] WD=0.120735  SWD=0.053928
[2800/124750] WD=0.162052  SWD=0.081616
[3000/124750] WD=0.224241  SWD=0.117044
[3200/124750] WD=0.099316  SWD=0.048049
[3400/124750] WD=0.047804  SWD=0.018087
[3600/124750] WD=0.093956  SWD=0.041381
[3800/124750] WD=0.058172  SWD=0.023051
[4000/124750] WD=0.130234  SWD=0.059522
[4200/124750] WD=0.093284  SWD=0.044925
[4400/124750] WD=0.171334  SWD=0.076426
[4600/124750] WD=0.017639  SWD=0.007069
[4800/124750] WD=0.

In [12]:
import numpy as np

WD_matrix = np.load('wasserstein_distances_full.npy')
done_mask = np.load('completed_mask.npy')

print("Nonzero non-NaN entries in WD_matrix:", np.count_nonzero(~np.isnan(WD_matrix) & (WD_matrix != 0)))
print("Pairs marked done in mask:", done_mask.sum() // 2)

Nonzero non-NaN entries in WD_matrix: 249500
Pairs marked done in mask: 124750
